# 1. Instalación de dependencias con resolución de conflictos
# Forzamos la actualización de LangChain y sus integraciones para evitar errores de compatibilidad

In [3]:
!pip install -q -U langchain langchain-community langchain-experimental langchain-mistralai pandas faiss-cpu tiktoken requests==2.32.4

import requests
import langchain_core
import langchain_experimental
import pandas as pd
import langchain

print(f"Instalación completada.")
print(f"Versión de requests: {requests.__version__}")
print(f"Versión de langchain-core: {langchain_core.__version__}")
print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de langchain: {langchain.__version__}")

Instalación completada.
Versión de requests: 2.32.4
Versión de langchain-core: 0.3.86
Versión de pandas: 2.2.2
Versión de langchain: 1.2.15


### 2. Normalización y Limpieza de Datos
Antes de pasar los datos a los agentes de IA, debemos asegurarnos de que estén limpios y en un formato estándar.

In [34]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Carga con codificación correcta
df_raw = pd.read_csv('sales_data_sample.csv', encoding='latin1')

# 2. Limpieza básica: nombres de columnas en minúsculas y sin espacios
df_raw.columns = df_raw.columns.str.strip().str.lower()

# 3. Conversión de tipos
if 'orderdate' in df_raw.columns:
    df_raw['orderdate'] = pd.to_datetime(df_raw['orderdate'])

# 4. Manejo de nulos (Solo si la columna existe)
if 'state' in df_raw.columns:
    df_raw['state'] = df_raw['state'].fillna('N/A')
else:
    print("Nota: La columna 'state' no se encontró en el dataset.")

# 5. Normalización de la columna 'sales' (Escalado entre 0 y 1)
if 'sales' in df_raw.columns:
    scaler = MinMaxScaler()
    df_raw['sales_normalized'] = scaler.fit_transform(df_raw[['sales']])

# Guardamos el dataframe limpio para los siguientes pasos
df = df_raw.copy()

print("Datos procesados y listos.")
display(df.head())

Datos procesados y listos.


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,productline,msrp,productcode,customername,phone,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize,sales_normalized
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small,0.175644
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,N/A,51100,France,EMEA,Henriot,Paul,Small,0.167916
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,N/A,75508,France,EMEA,Da Cunha,Daniel,Medium,0.250150
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium,0.240030
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium,0.347273


In [35]:
import pandas as pd
import os

# Usamos el archivo que el usuario subió localmente
file_path = 'sales_data_sample.csv'

if os.path.exists(file_path):
    try:
        # Cargar el archivo local
        df = pd.read_csv(file_path, encoding='latin1')

        # Normalizar nombres de columnas para que el bot no se confunda
        df.columns = df.columns.str.strip().str.lower()

        print(f"✅ ¡Éxito! Archivo local cargado correctamente.")
        print(f"Total de registros encontrados: {len(df)}")
        display(df.head(3))
    except Exception as e:
        print(f"❌ Error al leer el archivo local: {e}")
else:
    print(f"❌ No se encontró el archivo '{file_path}' en la carpeta de archivos (panel izquierdo).")

✅ ¡Éxito! Archivo local cargado correctamente.
Total de registros encontrados: 2823


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,productline,msrp,productcode,customername,phone,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium


## 🤖 Asistente de Ventas Maestro
Esta celda inicializa el chatbot híbrido que puede realizar cálculos complejos sobre el CSV y responder preguntas narrativas con memoria de conversación.

In [36]:
from langchain_core.callbacks.manager import handle_event
import os
import pandas as pd
import langchain
from google.colab import userdata
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_mistralai.embeddings import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# 1. Configuración de Identidad y LLM
api_key = userdata.get('API_KEY_MISTRAL')
llm = ChatMistralAI(mistral_api_key=api_key, model='open-mixtral-8x7b', temperature=0)

prefix = """
Eres un experto analista de datos de ventas.
El dataframe 'df' contiene 2,823 registros detallados.
Responde siempre de forma profesional y en español.
"""

try:
    # 2. Configurar RAG (Búsqueda de contexto)
    loader = CSVLoader(file_path='sales_data_sample.csv', encoding='latin1')
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    embeddings = MistralAIEmbeddings(mistral_api_key=api_key)
    vectorstore = FAISS.from_documents(splitter.split_documents(docs), embeddings)
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
    rag_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=vectorstore.as_retriever(), memory=memory)

    # 3. Configurar Agente de Pandas (Cálculos)
    agente_pandas = create_pandas_dataframe_agent(
        llm,
        df,
        prefix=prefix,
        verbose=False,
        allow_dangerous_code=True,
        handle_parsing_errors=True,
        agent_type="tool-calling"
    )

    print("✅ SISTEMA UNIFICADO LISTO")
except Exception as e:
    print(f"❌ Error en configuración: {e}")

✅ SISTEMA UNIFICADO LISTO


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


### 💬 Interfaz del Asistente
Ejecuta esta celda para iniciar la conversación con el experto en ventas.

In [37]:
print("====================================================")
print("🤖 ASISTENTE DE VENTAS MAESTRO ACTIVADO")
print("====================================================")
print("Escribe tu pregunta (o 'salir' para finalizar)")

while True:
    entrada = input("\n👉 Tu pregunta: ").strip()
    if not entrada: continue
    if entrada.lower() in ['salir', 'exit', 'quit']:
        print("👋 ¡Hasta pronto!")
        break

    # Enrutamiento inteligente
    keywords = ['cuanto', 'cuántos', 'promedio', 'suma', 'total', 'máximo', 'mínimo', 'calcular', 'ventas']
    usa_agente = any(word in entrada.lower() for word in keywords)

    try:
        if usa_agente:
            print("⏳ [Calculando datos numéricos...]")
            res = agente_pandas.invoke({"input": entrada})
            print(f"\n💬 {res['output']}")
        else:
            print("⏳ [Buscando en historial y contexto...]")
            res = rag_chain.invoke({"question": entrada})
            print(f"\n💬 {res['answer']}")
    except Exception as e:
        print(f"❌ Error: {e}")

🤖 ASISTENTE DE VENTAS MAESTRO ACTIVADO
Escribe tu pregunta (o 'salir' para finalizar)

👉 Tu pregunta: hola
⏳ [Buscando en historial y contexto...]

💬 ¡Hola! ¿En qué puedo ayudarte hoy?

👉 Tu pregunta: necesito saber cuántos autos tengo en stock
⏳ [Calculando datos numéricos...]

💬 Para determinar cuántos autos tienes en stock, necesito identificar qué columnas del dataframe `df` podrían representar la información de inventario o stock de productos. Sin embargo, en los datos que has compartido, no hay una columna explícita que indique la cantidad de autos en stock.

### Columnas relevantes en el dataframe:
1. **`productline`**: Indica la línea de producto, como "Motorcycles" (motocicletas).
2. **`productcode`**: Código del producto.
3. **`quantityordered`**: Cantidad ordenada en un pedido específico (no es stock).
4. **`msrp`**: Precio de venta al público sugerido (no es stock).
5. **`status`**: Estado del pedido (ej. "Shipped").

### Observaciones:
- No hay una columna como `stock` o `